# 08b — French Spam/Ham Detection Dataset

**C1 Source Type:** `Fichier de données (CSV / data file)`

---

## Objective

Load and normalize the **French Spam/Ham Detection** dataset from Kaggle
(`kinoux/french-spamham-detection-free`). This is a natively French dataset
with 1,000 labeled text samples.

### Dataset Info

| Field | Value |
|-------|-------|
| Source | [Kaggle: kinoux/french-spamham-detection-free](https://www.kaggle.com/datasets/kinoux/french-spamham-detection-free) |
| Format | JSONL file (downloaded via Kaggle CLI) |
| Rows | 1,000 |
| Columns | `text`, `label` |
| Labels | ham / spam |
| Language | French (native) |

### Pipeline

```
JSONL file → Load with pandas → Normalize schema → Export CSV
```

### Output

- `data/raw/csv/fr/french_spamham_<N>_<date>.csv`

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# ── Configuration ────────────────────────────────────────────────────
JSONL_PATH: Path = Path("data/raw/csv/french-spamham-detection-free/data.jsonl")
OUTPUT_DIR: Path = Path("data/raw/csv/fr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_TEXT_LENGTH: int = 20
SCHEMA_COLS: list[str] = ["text", "label", "source", "language"]

print(f"Input file   : {JSONL_PATH}")
print(f"Output dir   : {OUTPUT_DIR.resolve()}")
print(f"Min length   : {MIN_TEXT_LENGTH}")

Input file   : data/raw/csv/french-spamham-detection-free/data.jsonl
Output dir   : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/csv/fr
Min length   : 20


## 1. Load JSONL

In [2]:
# ── Load the JSONL file ──────────────────────────────────────────────
if not JSONL_PATH.exists():
    raise FileNotFoundError(
        f"JSONL not found at {JSONL_PATH}. "
        "Download via: kaggle datasets download kinoux/french-spamham-detection-free"
    )

df_raw: pd.DataFrame = pd.read_json(JSONL_PATH, lines=True)
print(f"Shape    : {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")
print(f"\nLabel distribution:")
print(df_raw["label"].value_counts())
df_raw.head(3)

Shape    : (1000, 2)
Columns  : ['text', 'label']

Label distribution:
label
ham     500
spam    500
Name: count, dtype: int64


,text,label
0,"Bien que prometteur, ce plateforme de développ...",ham
1,"DragDrop Pro, c'est cool pour construire des s...",ham
2,Je suis fort déçue par le service d'assistance...,ham


## 2. Normalize Schema

In [3]:
# ── Normalize to unified schema ──────────────────────────────────────
df_norm = pd.DataFrame({
    "text": df_raw["text"].astype(str),
    "label": df_raw["label"].apply(
        lambda x: "spam" if str(x).lower() in ("spam", "1") else "ham"
    ),
    "source": "kaggle_french_spamham",
    "language": "fr",
})

# Filter short texts
before: int = len(df_norm)
df_norm = df_norm[df_norm["text"].str.len() >= MIN_TEXT_LENGTH].reset_index(drop=True)
print(f"Before filter : {before:,}")
print(f"After filter  : {len(df_norm):,} (removed {before - len(df_norm)} short texts)")
print(f"\nLabel distribution:")
print(df_norm["label"].value_counts())
print(f"\nSample:")
df_norm.head(5)

Before filter : 1,000
After filter  : 1,000 (removed 0 short texts)

Label distribution:
label
ham     500
spam    500
Name: count, dtype: int64

Sample:


,text,label,source,language
0,"Bien que prometteur, ce plateforme de développ...",ham,kaggle_french_spamham,fr
1,"DragDrop Pro, c'est cool pour construire des s...",ham,kaggle_french_spamham,fr
2,Je suis fort déçue par le service d'assistance...,ham,kaggle_french_spamham,fr
3,Leur travail en branding est incroyable ! Ils ...,ham,kaggle_french_spamham,fr
4,Vérifiez immédiatement votre compte pour des a...,spam,kaggle_french_spamham,fr


## 3. Deduplicate

In [4]:
# ── Deduplication by text hash ────────────────────────────────────────
before = len(df_norm)
df_norm["text_hash"] = df_norm["text"].str[:300].apply(
    lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
)
df_norm = (
    df_norm.drop_duplicates(subset="text_hash", keep="first")
    .drop(columns="text_hash")
    .reset_index(drop=True)
)
after: int = len(df_norm)
print(f"Before dedup : {before:,}")
print(f"After dedup  : {after:,}")
print(f"Removed      : {before - after:,}")

Before dedup : 1,000
After dedup  : 1,000
Removed      : 0


## 4. Export

In [5]:
# ── Export ─────────────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
n_rows: int = len(df_norm)

if n_rows > 0:
    filename: str = f"french_spamham_{n_rows}_{timestamp}.csv"
    output_path: Path = OUTPUT_DIR / filename
    df_norm.to_csv(output_path, index=False, encoding="utf-8")

    size_kb: float = output_path.stat().st_size / 1024
    print(f"Exported  : {output_path}")
    print(f"Rows      : {n_rows:,}")
    print(f"Size      : {size_kb:.1f} KB")
    print(f"Columns   : {list(df_norm.columns)}")
else:
    print("Nothing to export.")

Exported  : data/raw/csv/fr/french_spamham_1000_20260301.csv
Rows      : 1,000
Size      : 161.1 KB
Columns   : ['text', 'label', 'source', 'language']


## 5. Summary

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Fichier de données — JSONL downloaded from Kaggle |
| **Provider** | `kinoux/french-spamham-detection-free` |
| **French content** | Natively French text (not translated) |
| **Schema** | Normalized to `(text, label, source, language)` |
| **Deduplication** | SHA-256 hash of first 300 chars |